In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

In [7]:
# Cargar variables de entorno
load_dotenv()

# --- Configuración ---
ES_URL = os.environ.get("ELASTICSEARCH_URL", "http://localhost:9200")
INDEX_NAME = "rag_index"

print(f"🔍 Conectando a Elasticsearch: {ES_URL}")
print(f"📚 Índice: {INDEX_NAME}\n")

🔍 Conectando a Elasticsearch: http://localhost:9200
📚 Índice: rag_index



In [8]:
# --- Embeddings ---
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# --- Cargar retriever desde Elasticsearch ---
try:
    vectorstore = ElasticsearchStore(
        es_url=ES_URL,
        index_name=INDEX_NAME,
        embedding=embeddings
    )
    print("✅ Conexión a Elasticsearch exitosa\n")
except Exception as e:
    print(f"❌ Error conectando a Elasticsearch: {e}")
    exit(1)

✅ Conexión a Elasticsearch exitosa



In [9]:
# --- Configurar retriever ---
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


In [10]:
# --- Query de prueba ---
query = "¿Cómo se gestiona un ciberincidente en una organización?"
print(f"🔎 Query: {query}\n")
print("=" * 80)

# --- Realizar búsqueda ---
try:
    results = retriever.invoke(query)
    print(f"\n✅ Se encontraron {len(results)} resultados\n")
    
    for i, doc in enumerate(results, 1):
        print(f"\n{'='*80}")
        print(f"📄 RESULTADO {i}")
        print(f"{'='*80}")
        print(f"📌 Sección: {doc.metadata.get('section_title', 'N/A')}")
        print(f"📖 Página: {doc.metadata.get('page', 'N/A')}")
        print(f"🔗 Chunk ID: {doc.metadata.get('chunk_id', 'N/A')}")
        print(f"📁 Source: {doc.metadata.get('source', 'N/A')}")
        print(f"\n💬 Texto:")
        print("-" * 80)
        print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)
        print("-" * 80)

except Exception as e:
    print(f"❌ Error en la búsqueda: {e}")

🔎 Query: ¿Cómo se gestiona un ciberincidente en una organización?


✅ Se encontraron 4 resultados


📄 RESULTADO 1
📌 Sección: 7.3. CONTENCIÓN
📖 Página: 31
🔗 Chunk ID: fafc13e0-16a4-41f0-afe2-f20edfc958f8
📁 Source: guia_gestion_ciberincidentes .pdf

💬 Texto:
--------------------------------------------------------------------------------
En el momento que se ha identificado un ciberincidente la máxima prioridad es contener el impacto del mismo en la organización de forma que se puedan evitar lo antes posible la propagación a otros sistemas o redes evitando un impacto mayor, y la extracción de información fuera de la organización.
--------------------------------------------------------------------------------

📄 RESULTADO 2
📌 Sección: Tabla 9. Tiempos de cierre del ciberincidente sin respuesta
📖 Página: 29
🔗 Chunk ID: edc2c9fc-b410-40fe-9eee-d1831fc07c56
📁 Source: guia_gestion_ciberincidentes .pdf

💬 Texto:
--------------------------------------------------------------------------------


In [11]:
# --- Estadísticas adicionales ---
print(f"\n{'='*80}")
print("📊 ESTADÍSTICAS")
print(f"{'='*80}")
print(f"Total de resultados: {len(results)}")
print(f"Longitud promedio: {sum(len(doc.page_content) for doc in results) / len(results):.0f} caracteres")

# --- Verificar diversidad de fuentes ---
sources = set(doc.metadata.get('source', 'N/A') for doc in results)
sections = set(doc.metadata.get('section_title', 'N/A') for doc in results)
pages = set(doc.metadata.get('page', 'N/A') for doc in results)

print(f"\nFuentes únicas: {len(sources)}")
print(f"Secciones únicas: {len(sections)}")
print(f"Páginas únicas: {len(pages)}")


📊 ESTADÍSTICAS
Total de resultados: 4
Longitud promedio: 363 caracteres

Fuentes únicas: 1
Secciones únicas: 4
Páginas únicas: 4


In [ ]:
async def respond(
    state: State, *, config: RunnableConfig
) -> dict[str, list[BaseMessage]]:
    """Call the LLM powering our "agent"."""
    configuration = Configuration.from_runnable_config(config)
    # Feel free to customize the prompt, model, and other logic!
    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", configuration.response_system_prompt),
            ("placeholder", "{messages}"),
        ]
    )
    model = load_chat_model(configuration.response_model).bind(
        temperature=configuration.response_temperature
    )

    retrieved_docs = format_docs(state.retrieved_docs)
    message_value = await prompt.ainvoke(
        {
            "messages": state.messages,
            "retrieved_docs": retrieved_docs,
            "system_time": datetime.now(tz=timezone.utc).isoformat(),
        },
        config,
    )
    response = await model.ainvoke(message_value, config)
    # We return a list, because this will get added to the existing list
    return {"messages": [response]}

In [ ]:
builder = StateGraph(State, input_schema=InputState, context_schema=Configuration)

builder.add_node(generate_query)  # type: ignore[arg-type]
builder.add_node(retrieve)  # type: ignore[arg-type]
builder.add_node(respond)  # type: ignore[arg-type]
builder.add_edge("__start__", "generate_query")
builder.add_edge("generate_query", "retrieve")
builder.add_edge("retrieve", "respond")

# Finally, we compile it!
# This compiles it into a graph you can invoke and deploy.
graph = builder.compile(
    interrupt_before=[],  # if you want to update the state before calling the tools
    interrupt_after=[],
)
graph.name = "RetrievalGraph"

In [ ]:
"""Default prompts."""

RESPONSE_SYSTEM_PROMPT = """You are a helpful AI assistant. Answer the user's questions based on the retrieved documents.

{retrieved_docs}

System time: {system_time}"""
QUERY_SYSTEM_PROMPT = """Generate search queries to retrieve documents that may help answer the user's question. Previously, you made the following queries:
    
<previous_queries/>
{queries}
</previous_queries>

System time: {system_time}"""